# Ablation Study — GCN, GAT, SchNet on QM9

Controlled single-variable experiments: vary one hyperparameter at a time
while holding all others at their default values.

**Ablations:**
We compare four architectures: **GCN** (uniform neighbor aggregation),
**GAT** (learned attention over neighbors using only node features),
**GATv2** (dynamic attention that also incorporates **edge attributes** like bond type into the attention score),
and **SchNet** (continuous-filter convolutions on 3D atomic coordinates).
Including both GAT and GATv2 isolates the effect of edge-aware attention.

1. Depth (num\_layers): 1, 2, 4, 6, 8
2. Width (hidden\_dim): 32, 64, 128, 256
3. Feature mode: topology vs full
4. Dropout: 0.0, 0.1, 0.2, 0.3
5. Pooling (SchNet): add vs mean
6. Target property: mu, U0, Cv

**Budget:** 50 epochs per trial, patience 10 (enough for relative trends).

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE_ROOT  = '/content/drive/MyDrive/gnn_qm9'
PROJECT_DIR = '/content/gnn_qm9'
REPO_URL    = 'https://github.com/amanikonda123/DL-Final-Project.git'

os.makedirs(f'{DRIVE_ROOT}/outputs/plots/ablation', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/outputs/results', exist_ok=True)

if os.path.exists(PROJECT_DIR):
    !git -C {PROJECT_DIR} pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

TORCH_VER = torch.__version__.split('\+')[0]
CUDA_VER  = 'cu121' if torch.cuda.is_available() else 'cpu'
!pip install -q torch-geometric
!pip install -q pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html
!pip install -q pyyaml tqdm

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT  = f'{DRIVE_ROOT}/data/qm9_raw'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'
PLOT_DIR   = f'{OUTPUT_DIR}/plots/ablation'
print(f'Device: {DEVICE}')

In [ ]:
# ── Cell 2: Ablation Runner ──────────────────────────────────────────────────
import copy, yaml, csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from train import train_model
from data.loader import get_dataloaders

ABLATION_EPOCHS   = 50
ABLATION_PATIENCE = 10

def run_ablation(base_cfg, model_name, param_path, values, loaders=None):
    """
    Run a single-variable ablation.

    Args:
        base_cfg:    base config dict
        model_name:  "gcn", "gat", or "schnet"
        param_path:  dot-separated path e.g. "model.num_layers"
        values:      list of values to sweep
        loaders:     optional (train_loader, val_loader, normalizer) tuple

    Returns:
        pd.DataFrame with columns [param_value, best_val_mae, model]
    """
    keys = param_path.split(".")
    rows = []

    for val in values:
        cfg = copy.deepcopy(base_cfg)
        cfg["training"]["epochs"]   = ABLATION_EPOCHS
        cfg["training"]["patience"] = ABLATION_PATIENCE

        # Set the ablation parameter
        d = cfg
        for k in keys[:-1]:
            d = d[k]
        d[keys[-1]] = val

        # For GAT/GCN, ensure hidden_dim divisible by heads
        if model_name in ("gat", "gatv2"):
            heads = cfg["model"].get("heads", 4)
            hdim  = cfg["model"].get("hidden_dim", 128)
            if hdim % heads != 0:
                print(f"  skip {param_path}={val} (hidden_dim {hdim} % heads {heads} != 0)")
                continue

        print(f"  {model_name} | {param_path}={val}")
        try:
            result = train_model(
                cfg, device=str(DEVICE),
                output_dir=f"{OUTPUT_DIR}/ablation_tmp",
                data_root=DATA_ROOT,
                model_name=model_name,
                loaders=loaders,
            )
            rows.append({
                "param_value": val,
                "best_val_mae": result["best_val_mae"],
                "model": model_name,
            })
        except Exception as e:
            print(f"    FAILED: {e}")

    return pd.DataFrame(rows)


def load_shared_loaders(cfg):
    """Load data once and return (train_loader, val_loader, normalizer)."""
    train_ld, val_ld, _, norm = get_dataloaders(cfg, root=DATA_ROOT)
    return (train_ld, val_ld, norm)

## Ablation 1: Depth (num\_layers)

**Hypothesis:** Deeper GCN suffers from over-smoothing (node embeddings converge),
while GAT's attention mechanism should resist this. SchNet's interaction blocks
are distance-based so depth effects may differ.

In [ ]:
# ── Cell 3: Depth Ablation ───────────────────────────────────────────────────
DEPTH_VALUES = [1, 2, 4, 6, 8]

# Load base configs
with open("config/gcn.yaml") as f: gcn_cfg = yaml.safe_load(f)
with open("config/gat.yaml") as f: gat_cfg = yaml.safe_load(f)
with open("config/schnet.yaml") as f: schnet_cfg = yaml.safe_load(f)

# Shared loaders (topology mode for GCN/GAT)
topo_loaders = load_shared_loaders(gcn_cfg)
geo_loaders  = load_shared_loaders(schnet_cfg)

print("=== Depth Ablation ===")
depth_gcn    = run_ablation(gcn_cfg,    "gcn",    "model.num_layers", DEPTH_VALUES, loaders=topo_loaders)
depth_gat    = run_ablation(gat_cfg,    "gat",    "model.num_layers", DEPTH_VALUES, loaders=topo_loaders)
depth_schnet = run_ablation(schnet_cfg, "schnet", "model.num_interactions", DEPTH_VALUES, loaders=geo_loaders)

depth_df = pd.concat([depth_gcn, depth_gat, depth_schnet], ignore_index=True)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
for name, group in depth_df.groupby("model"):
    ax.plot(group["param_value"], group["best_val_mae"], "o-", label=name.upper(), linewidth=2, markersize=6)
ax.set_xlabel("Number of Layers / Interactions", fontsize=12)
ax.set_ylabel("Best Val MAE (Ha)", fontsize=12)
ax.set_title("Depth Ablation — Effect of Network Depth", fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xticks(DEPTH_VALUES)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/depth_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {PLOT_DIR}/depth_ablation.png")

## Ablation 2: Width (hidden\_dim)

**Hypothesis:** Larger hidden dimensions increase model capacity but risk overfitting
on the relatively simple QM9 molecules. There may be diminishing returns.

In [ ]:
# ── Cell 4: Width Ablation ───────────────────────────────────────────────────
WIDTH_VALUES = [32, 64, 128, 256]

print("=== Width Ablation ===")
width_gcn    = run_ablation(gcn_cfg,    "gcn",    "model.hidden_dim", WIDTH_VALUES, loaders=topo_loaders)
width_gat    = run_ablation(gat_cfg,    "gat",    "model.hidden_dim", WIDTH_VALUES, loaders=topo_loaders)

# SchNet uses hidden_channels
width_schnet = run_ablation(schnet_cfg, "schnet", "model.hidden_channels", WIDTH_VALUES, loaders=geo_loaders)

width_df = pd.concat([width_gcn, width_gat, width_schnet], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 5))
for name, group in width_df.groupby("model"):
    ax.plot(group["param_value"], group["best_val_mae"], "o-", label=name.upper(), linewidth=2, markersize=6)
ax.set_xlabel("Hidden Dimension", fontsize=12)
ax.set_ylabel("Best Val MAE (Ha)", fontsize=12)
ax.set_title("Width Ablation — Effect of Hidden Size", fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xticks(WIDTH_VALUES)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/width_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## Ablation 3: Feature Mode (topology vs full)

**Hypothesis:** Using all 11 node features (full mode) instead of just the 5-dim atom
type one-hot (topology mode) should help GCN and GAT by providing richer chemical info
(hybridization, hydrogen count, formal charge).

In [ ]:
# ── Cell 5: Feature Mode Ablation ────────────────────────────────────────────
FEAT_VALUES = ["topology", "full"]

print("=== Feature Mode Ablation ===")
feat_gcn = run_ablation(gcn_cfg, "gcn", "dataset.feature_mode", FEAT_VALUES)
feat_gat = run_ablation(gat_cfg, "gat", "dataset.feature_mode", FEAT_VALUES)

feat_df = pd.concat([feat_gcn, feat_gat], ignore_index=True)

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(FEAT_VALUES))
width = 0.3
for i, (name, group) in enumerate(feat_df.groupby("model")):
    vals = [group[group["param_value"] == v]["best_val_mae"].values[0]
            if len(group[group["param_value"] == v]) > 0 else 0
            for v in FEAT_VALUES]
    ax.bar(x + i * width, vals, width, label=name.upper())
ax.set_xlabel("Feature Mode", fontsize=12)
ax.set_ylabel("Best Val MAE (Ha)", fontsize=12)
ax.set_title("Feature Mode Ablation — Topology vs Full Node Features", fontsize=13)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(FEAT_VALUES)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/feature_mode_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## Ablation 4: Dropout

**Hypothesis:** Some dropout should help regularize, but too much will hurt
convergence on QM9's regression targets. The optimal amount may differ between
GCN (which has residual connections) and GAT (which has attention coefficients).

In [ ]:
# ── Cell 6: Dropout Ablation ─────────────────────────────────────────────────
DROP_VALUES = [0.0, 0.1, 0.2, 0.3]

print("=== Dropout Ablation ===")
drop_gcn = run_ablation(gcn_cfg, "gcn", "model.dropout", DROP_VALUES, loaders=topo_loaders)
drop_gat = run_ablation(gat_cfg, "gat", "model.dropout", DROP_VALUES, loaders=topo_loaders)

drop_df = pd.concat([drop_gcn, drop_gat], ignore_index=True)

fig, ax = plt.subplots(figsize=(8, 5))
for name, group in drop_df.groupby("model"):
    ax.plot(group["param_value"], group["best_val_mae"], "o-", label=name.upper(), linewidth=2, markersize=6)
ax.set_xlabel("Dropout Rate", fontsize=12)
ax.set_ylabel("Best Val MAE (Ha)", fontsize=12)
ax.set_title("Dropout Ablation — Effect of Regularization", fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/dropout_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## Ablation 5: Pooling Strategy (SchNet)

**Hypothesis:** Additive readout (sum) preserves extensive molecular properties
like total energy, while mean readout normalizes by molecule size.
U0 is an extensive property so sum should perform better.

In [ ]:
# ── Cell 7: Pooling Ablation (SchNet) ────────────────────────────────────────
POOL_VALUES = ["add", "mean"]

print("=== Pooling Ablation (SchNet) ===")
pool_schnet = run_ablation(schnet_cfg, "schnet", "model.readout", POOL_VALUES, loaders=geo_loaders)

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(pool_schnet["param_value"], pool_schnet["best_val_mae"],
       color=["steelblue", "coral"], width=0.5)
ax.set_xlabel("Readout Strategy", fontsize=12)
ax.set_ylabel("Best Val MAE (Ha)", fontsize=12)
ax.set_title("SchNet Pooling Ablation — Add vs Mean Readout", fontsize=13)
ax.grid(axis="y", alpha=0.3)
for i, row in pool_schnet.iterrows():
    ax.text(i, row["best_val_mae"] * 1.02, f'{row["best_val_mae"]:.2f}',
            ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/pooling_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## Ablation 6: Target Property

**Hypothesis:** Different molecular properties have different difficulty levels.
Dipole moment (mu) depends on charge distribution, U0 on total energy, and Cv on
vibrational modes. Geometry-aware SchNet should excel on all, while topology-only
models may struggle more on geometry-sensitive properties.

In [ ]:
# ── Cell 8: Target Property Ablation ─────────────────────────────────────────
TARGET_MAP = {0: "mu (Dipole)", 7: "U0 (Energy)", 11: "Cv (Heat cap)"}
TARGET_VALUES = list(TARGET_MAP.keys())

print("=== Target Property Ablation ===")
# Need fresh loaders per target since normalization differs
tgt_gcn_rows, tgt_gat_rows, tgt_schnet_rows = [], [], []

for tgt in TARGET_VALUES:
    print(f"
--- Target {tgt}: {TARGET_MAP[tgt]} ---")

    gcfg = copy.deepcopy(gcn_cfg)
    gcfg["dataset"]["target"] = tgt
    gcfg["training"]["epochs"] = ABLATION_EPOCHS
    gcfg["training"]["patience"] = ABLATION_PATIENCE
    topo_ld = load_shared_loaders(gcfg)

    scfg = copy.deepcopy(schnet_cfg)
    scfg["dataset"]["target"] = tgt
    scfg["training"]["epochs"] = ABLATION_EPOCHS
    scfg["training"]["patience"] = ABLATION_PATIENCE
    geo_ld = load_shared_loaders(scfg)

    for name, cfg, ld, row_list in [
        ("gcn", gcfg, topo_ld, tgt_gcn_rows),
        ("gat", copy.deepcopy(gat_cfg), topo_ld, tgt_gat_rows),
        ("schnet", scfg, geo_ld, tgt_schnet_rows),
    ]:
        if name == "gat":
            cfg["dataset"]["target"] = tgt
            cfg["training"]["epochs"] = ABLATION_EPOCHS
            cfg["training"]["patience"] = ABLATION_PATIENCE
        result = train_model(
            cfg, device=str(DEVICE),
            output_dir=f"{OUTPUT_DIR}/ablation_tmp",
            data_root=DATA_ROOT, model_name=name, loaders=ld,
        )
        row_list.append({
            "param_value": TARGET_MAP[tgt],
            "best_val_mae": result["best_val_mae"],
            "model": name,
        })

tgt_df = pd.DataFrame(tgt_gcn_rows + tgt_gat_rows + tgt_schnet_rows)

fig, ax = plt.subplots(figsize=(10, 5))
models = ["gcn", "gat", "schnet"]
x = np.arange(len(TARGET_MAP))
width = 0.25
for i, m in enumerate(models):
    sub = tgt_df[tgt_df["model"] == m]
    ax.bar(x + i * width, sub["best_val_mae"].values, width, label=m.upper())
ax.set_xlabel("Target Property", fontsize=12)
ax.set_ylabel("Best Val MAE", fontsize=12)
ax.set_title("Target Property Ablation — Task Difficulty per Architecture", fontsize=13)
ax.set_xticks(x + width)
ax.set_xticklabels(list(TARGET_MAP.values()))
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/target_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Cell 9: Summary Table ────────────────────────────────────────────────────
print("
" + "=" * 60)
print("ABLATION STUDY SUMMARY")
print("=" * 60)

all_dfs = {
    "depth": depth_df,
    "width": width_df,
    "feature_mode": feat_df,
    "dropout": drop_df,
    "pooling": pool_schnet,
    "target": tgt_df,
}

for name, df in all_dfs.items():
    print(f"
--- {name.upper()} ---")
    for model_name, group in df.groupby("model"):
        best = group.loc[group["best_val_mae"].idxmin()]
        print(f"  {model_name.upper():>8s}: best={best["param_value"]:>10}  MAE={best["best_val_mae"]:.4f}")

# Save all results to CSV
summary = pd.concat([df.assign(ablation=name) for name, df in all_dfs.items()], ignore_index=True)
summary_path = f"{OUTPUT_DIR}/results/ablation_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"
Full results saved to {summary_path}")
print(f"All plots saved to {PLOT_DIR}/")